## Container lifecycle — start / stop / rm

A container has a small set of states and a handful of commands that move it between them. `docker run` fuses the first two steps (`create` + `start`), but pulling them apart shows the whole map:

```
  docker create      +---------+  docker start   +---------+
  image ───────────► | created | ──────────────► | running |
                     +---------+                 +----+----+
                                                      │ docker stop
                                    docker start      │ (SIGTERM → SIGKILL)
                        +---------+ ◄────────────      ▼
                        | stopped | ◄──────────── (process exits)
                        +----+----+
                             │ docker rm
                             ▼
                          removed
```

The everyday verbs:

- **`docker stop <c>`** — graceful shutdown: the daemon sends **SIGTERM**, waits (10s by default), then **SIGKILL** if the process ignored it. Well-behaved programs use that window to flush and close cleanly.
- **`docker start <c>`** — bring a stopped container back to running, with its writable layer intact.
- **`docker restart <c>`** — stop then start in one step.
- **`docker rm <c>`** — delete a *stopped* container (its writable layer and all). Add `-f` to force-remove a running one (SIGKILL + remove).
- **`docker kill <c>`** — immediate SIGKILL, no grace period.

**Exit codes tell you how it ended.** `docker ps -a` shows e.g. `Exited (137)`. The Unix convention: `0` = clean success; `125` = the `docker run` command itself failed; `126/127` = the container's command wasn't executable / wasn't found; `137` = killed by SIGKILL (often out-of-memory); `143` = killed by SIGTERM (a clean `docker stop`). The `128 + signal` math (`137 = 128 + 9`) is Linux, and Docker surfaces it faithfully — a fast way to read *why* a container died.